# 06 — Classification Data QA

Visualises tumor patches extracted using **GT masks** vs **predicted masks** side-by-side.
Run this before training (NB07) to confirm the ROI extraction looks correct.

**No Drive writes — purely read-only QA.** No sync cell needed.

**Prerequisites:**
- `data/figshare/processed/` on Drive (NB01)
- `data/figshare/splits/<scheme>/` on Drive (NB02)
- Seg predictions on Drive: `outputs/predictions/segmentation/figshare/<SEG_EXPERIMENT>/fold_X/` (NB05)

**Knobs (Cell 3):**
- `DATASET` — dataset to QA
- `SEG_EXPERIMENT` — seg experiment whose predicted masks to compare against GT
- `SPLIT_SCHEME` — fold scheme
- `FOLD` — which fold's test set to sample from
- `N_SAMPLES` — number of images to display

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists('/content/senior_project'):
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = 'https://github.com/salemaker47/senior_project.git'
    if token:
        url = url.replace('https://', f'https://{token}@', 1)
    os.system(f'git clone {url} /content/senior_project')
if '/content/senior_project' not in sys.path:
    sys.path.insert(0, '/content/senior_project')

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url='https://github.com/salemaker47/senior_project.git',
)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')

## Cell 3 — Config

Set `SEG_EXPERIMENT` to whichever seg experiment's predicted masks you want to compare against GT.
Use the same value you'll pass as `SEG_EXPERIMENT` in NB08.

In [ ]:
DATASET       = 'figshare'
SEG_EXPERIMENT = '07_unetpp_effb4_dicebce_image_level'   # seg experiment for predicted masks
SPLIT_SCHEME  = 'image_level'
FOLD          = 1      # which fold's test set to sample from
N_SAMPLES     = 12     # number of images to show

assert DATASET in ('figshare',), f'Only figshare is supported for classification, got {DATASET!r}'
print(f'Dataset:       {DATASET}')
print(f'Seg experiment: {SEG_EXPERIMENT}')
print(f'Fold:          {FOLD}  (split_scheme={SPLIT_SCHEME})')
print(f'N_SAMPLES:     {N_SAMPLES}')

## Cell 4 — Copy data to local SSD

In [ ]:
import shutil
from src.notebook_setup import copy_to_local
from src.file_utils     import fold_split_csv_paths, seg_predictions_dir

LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[DATASET])

# Copy seg predictions for the chosen fold to local SSD
drive_preds = DRIVE_ROOT / 'outputs' / 'predictions' / 'segmentation' / DATASET / SEG_EXPERIMENT
local_preds = LOCAL_ROOT / 'outputs' / 'predictions' / 'segmentation' / DATASET / SEG_EXPERIMENT
if drive_preds.exists():
    local_preds.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(drive_preds / f'fold_{FOLD}'), str(local_preds / f'fold_{FOLD}'), dirs_exist_ok=True)
    print(f'Copied seg predictions fold {FOLD} to local SSD')
else:
    print(f'WARNING: seg predictions not found at {drive_preds}')
    print('         Predicted-mask panel will be unavailable.')

# Load fold test CSV
from src.data_utils import load_metadata
csv_paths = fold_split_csv_paths(LOCAL_ROOT, dataset=DATASET, scheme=SPLIT_SCHEME, fold=FOLD)
test_df = load_metadata(csv_paths['test'])
print(f'Test set: {len(test_df)} images, {test_df["tumor_class"].value_counts().to_dict()}')

## Cell 5 — Side-by-side patch visualisation

For each sampled image: original MRI | GT mask | GT patch | predicted mask | predicted patch

In [ ]:
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

from src.cls_data_utils import extract_patch, CLASS_TO_IDX
from src.file_utils     import seg_predictions_dir

random.seed(42)
sample_df = test_df.sample(min(N_SAMPLES, len(test_df)), random_state=42).reset_index(drop=True)

pred_fold_dir = local_preds / f'fold_{FOLD}' if (local_preds / f'fold_{FOLD}').exists() else None

n_cols = 5  # MRI | GT mask | GT patch | pred mask | pred patch
n_rows = len(sample_df)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
if n_rows == 1:
    axes = axes[np.newaxis, :]

col_labels = ['MRI', 'GT mask', 'GT patch', 'Pred mask', 'Pred patch']
for col_idx, lbl in enumerate(col_labels):
    axes[0, col_idx].set_title(lbl, fontsize=10)

for row_idx, (_, row) in enumerate(sample_df.iterrows()):
    img_path  = LOCAL_ROOT / row['image_path']
    mask_path = LOCAL_ROOT / row['mask_path']
    image_id  = str(row['image_id'])
    cls_label = str(row.get('tumor_class', '?'))

    img  = cv2.imread(str(img_path),  cv2.IMREAD_GRAYSCALE)
    mask_gt = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask_gt_bin = (mask_gt > 127).astype(np.uint8) * 255 if mask_gt is not None else None

    # Predicted mask
    pred_mask_bin = None
    if pred_fold_dir is not None:
        pred_path = pred_fold_dir / f'{image_id}.png'
        if pred_path.exists():
            pm = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE)
            pred_mask_bin = (pm > 127).astype(np.uint8) * 255 if pm is not None else None

    gt_patch   = extract_patch(img, mask_gt_bin,   target_size=224) if (img is not None and mask_gt_bin is not None) else None
    pred_patch = extract_patch(img, pred_mask_bin, target_size=224) if (img is not None and pred_mask_bin is not None) else None

    def show(ax, data, cmap=None, title_suffix=''):
        ax.set_xlabel(f'{cls_label}  {image_id}{title_suffix}', fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
        if data is None:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        else:
            ax.imshow(data, cmap=cmap)

    show(axes[row_idx, 0], img,          cmap='gray')
    show(axes[row_idx, 1], mask_gt_bin,  cmap='gray')
    show(axes[row_idx, 2], gt_patch)
    show(axes[row_idx, 3], pred_mask_bin, cmap='gray', title_suffix=' (pred)')
    show(axes[row_idx, 4], pred_patch,    title_suffix=' (pred patch)')

fig.suptitle(
    f'{DATASET} fold {FOLD} test set — GT vs predicted patches\n'
    f'seg: {SEG_EXPERIMENT}',
    fontsize=11
)
fig.tight_layout()
plt.show()
print(f'Displayed {n_rows} samples from fold {FOLD} test set.')

## Cell 6 — Class distribution

In [ ]:
import pandas as pd
from src.data_utils import load_metadata
from src.file_utils import fold_split_csv_paths

print(f'Fold {FOLD} splits ({SPLIT_SCHEME}):')
for split in ('train', 'val', 'test'):
    df = load_metadata(fold_split_csv_paths(LOCAL_ROOT, DATASET, SPLIT_SCHEME, FOLD)[split])
    counts = df['tumor_class'].value_counts().to_dict()
    print(f'  {split:5s}: {len(df):>5} images  |  {counts}')